# M-04 Temporal Fall Detection

M-03이 생성한 사람별 pose keypoint 시퀀스를 입력으로 받아 낙상 의심 여부를 판단하는 실험 Notebook

## 진행 순서

1. 환경 설정
   - Colab 실행 환경과 Google Drive 경로 확인
   - GPU와 PyTorch 환경 확인
   - M-04 실행 package 설치
2. M-03 keypoint sequence 입력
   - `track_id`, `timestamp_ms`, COCO-17 keypoint와 confidence 확인
   - 누락 frame과 낮은 confidence 처리
3. 낙상 dataset 준비
   - URFD RGB·label 다운로드와 무결성 확인
   - M-03 pipeline을 이용한 keypoint sequence 생성
   - subject와 원본 sequence 기준 train·validation·test 분리
4. 규칙 기반 낙상 판단 baseline
   - 하강 속도, 몸통 회전, 낮은 자세 지속과 회복 특징 계산
   - 2초·3초 window와 threshold 비교
5. 경량 temporal model 비교
   - 1D CNN, TCN, 소형 GRU 중 1~2개 비교
   - 규칙 baseline과 같은 split·지표 사용
6. event 평가와 M-04 출력 형식 확인
   - precision, recall, F1, 시간당 오경보와 감지 지연 측정
   - `fall_suspected`, confidence와 판단 근거 출력

# 1. 환경 설정

Colab과 로컬 Jupyter에서 동일한 project root를 찾고, Git에서 제외되는 raw dataset 경로를 준비

In [1]:
import importlib.util
import os
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    DRIVE_MOUNT_POINT = Path("/content/drive")
    drive.mount(str(DRIVE_MOUNT_POINT))
    PROJECT_ROOT = (
        DRIVE_MOUNT_POINT / "MyDrive" / "Colab Notebooks" / "wardy-pose-fall"
    )
    runtime_name = "Google Colab"
else:
    def find_project_root(start_path: Path) -> Path:
        for candidate in (start_path, *start_path.parents):
            if (candidate / ".git").exists() and (candidate / "ml" / "notebook").is_dir():
                return candidate
        raise FileNotFoundError(
            "wardy 저장소 내부에서 Jupyter 실행 또는 WARDY_PROJECT_ROOT 설정 필요"
        )

    search_start = Path(
        os.environ.get("WARDY_PROJECT_ROOT", Path.cwd())
    ).expanduser().resolve()
    PROJECT_ROOT = find_project_root(search_start)
    runtime_name = "Local Jupyter"

RAW_DATA_DIR = PROJECT_ROOT / "ml" / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime: {runtime_name}")
print(f"Project repository: {PROJECT_ROOT.name}")
print(f"Raw dataset directory: {RAW_DATA_DIR.relative_to(PROJECT_ROOT)}")

Runtime: Local Jupyter
Project repository: wardy-pose-fall
Raw dataset directory: ml/data/raw


# 3. 낙상 dataset 준비

## URFD RGB sample 다운로드

공식 출처: https://fenix.ur.edu.pl/~mkepski/ds/uf.html

정면 camera인 `fall-01-cam0-rgb.zip` 하나만 내려받아 낙상 sequence 전처리 확인

In [2]:
import hashlib
import shutil
from urllib.request import urlopen
from zipfile import ZipFile

URFD_DIR = RAW_DATA_DIR / "urfd"
SAMPLE_URL = "https://fenix.ur.edu.pl/~mkepski/ds/data/fall-01-cam0-rgb.zip"
SAMPLE_ARCHIVE = URFD_DIR / "fall-01-cam0-rgb.zip"
PARTIAL_ARCHIVE = SAMPLE_ARCHIVE.with_suffix(".zip.part")
SAMPLE_SHA256 = "e5ff96108d322d7fe5ba6871a4c50d9c26c59e74abea03d2b69b6be56521736f"

URFD_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if SAMPLE_ARCHIVE.exists() and sha256(SAMPLE_ARCHIVE) == SAMPLE_SHA256:
    archive_state = "cached"
else:
    try:
        with urlopen(SAMPLE_URL, timeout=60) as response, PARTIAL_ARCHIVE.open("wb") as output:
            shutil.copyfileobj(response, output)
        downloaded_hash = sha256(PARTIAL_ARCHIVE)
        if downloaded_hash != SAMPLE_SHA256:
            raise RuntimeError(f"Archive SHA-256 mismatch: {downloaded_hash}")
        PARTIAL_ARCHIVE.replace(SAMPLE_ARCHIVE)
    finally:
        PARTIAL_ARCHIVE.unlink(missing_ok=True)
    archive_state = "downloaded"

archive_hash = sha256(SAMPLE_ARCHIVE)

with ZipFile(SAMPLE_ARCHIVE) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise ValueError(f"Corrupted ZIP member: {bad_member}")
    png_count = sum(name.lower().endswith(".png") for name in archive.namelist())

print(f"Archive state: {archive_state}")
print(f"Archive: {SAMPLE_ARCHIVE}")
print(f"SHA-256: {archive_hash}")
print(f"Size: {SAMPLE_ARCHIVE.stat().st_size / 1024**2:.2f} MiB")
print(f"PNG frames: {png_count}")

Archive state: cached
Archive: /Users/mumallaeng/git/wardy-pose-fall/ml/data/raw/urfd/fall-01-cam0-rgb.zip
SHA-256: e5ff96108d322d7fe5ba6871a4c50d9c26c59e74abea03d2b69b6be56521736f
Size: 55.05 MiB
PNG frames: 160
